# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [3]:
!pip install -U imbalanced-learn



   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]



In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


In [12]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


In [13]:
# Step 1: Check target variable distribution
target_counts = fraud['fraud'].value_counts()
target_pct = fraud['fraud'].value_counts(normalize=True) * 100

print("\n Target Variable Distribution ")
for val, count in target_counts.items():
    print(f"Class {val} ({'Legit' if val==0 else 'Fraud'}): {count} samples ({target_pct[val]:.2f}%)")

print("\nConclusion Step 1: Yes, we are dealing with a highly imbalanced dataset. Fraudulent transactions make up less than 9% of the total data pool.")



 Target Variable Distribution 
Class 0.0 (Legit): 912597 samples (91.26%)
Class 1.0 (Fraud): 87403 samples (8.74%)

Conclusion Step 1: Yes, we are dealing with a highly imbalanced dataset. Fraudulent transactions make up less than 9% of the total data pool.


In [14]:
# Prepare Data Splits
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
from sklearn.linear_model import LogisticRegression
# Explicitly import all missing evaluation functions
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Step 2: Train Logistic Regression baseline
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

# Step 3: Evaluate model
y_pred_base = baseline_model.predict(X_test)
y_prob_base = baseline_model.predict_proba(X_test)[:, 1]

print("--- Baseline Classification Report (Imbalanced Data) ---")
print(classification_report(y_test, y_pred_base))
print(f"Baseline ROC-AUC Score: {roc_auc_score(y_test, y_prob_base):.4f}")
print("\nBaseline Confusion Matrix:\n", confusion_matrix(y_test, y_pred_base))


--- Baseline Classification Report (Imbalanced Data) ---
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182557
         1.0       0.89      0.60      0.72     17443

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.95    200000

Baseline ROC-AUC Score: 0.9670

Baseline Confusion Matrix:
 [[181280   1277]
 [  6991  10452]]


In [22]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Run Oversample
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

# Retrain and Evaluate
model_over = LogisticRegression(max_iter=1000, random_state=42)
model_over.fit(X_train_over, y_train_over)

y_pred_over = model_over.predict(X_test)
y_prob_over = model_over.predict_proba(X_test)[:, 1]

print("\n--- Oversampled Classification Report ---")
print(classification_report(y_test, y_pred_over))
print(f"Oversampled ROC-AUC Score: {roc_auc_score(y_test, y_prob_over):.4f}")



--- Oversampled Classification Report ---
              precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Oversampled ROC-AUC Score: 0.9796


In [23]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Run Undersample
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

# Retrain and Evaluate
model_under = LogisticRegression(max_iter=1000, random_state=42)
model_under.fit(X_train_under, y_train_under)

y_pred_under = model_under.predict(X_test)
y_prob_under = model_under.predict_proba(X_test)[:, 1]

print("\n--- Undersampled Classification Report ---")
print(classification_report(y_test, y_pred_under))
print(f"Undersampled ROC-AUC Score: {roc_auc_score(y_test, y_prob_under):.4f}")



--- Undersampled Classification Report ---
              precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

Undersampled ROC-AUC Score: 0.9795


In [24]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Run SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Retrain and Evaluate
model_smote = LogisticRegression(max_iter=1000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test)
y_prob_smote = model_smote.predict_proba(X_test)[:, 1]

print("\n--- SMOTE Sampling Classification Report ---")
print(classification_report(y_test, y_pred_smote))
print(f"SMOTE ROC-AUC Score: {roc_auc_score(y_test, y_prob_smote):.4f}")



--- SMOTE Sampling Classification Report ---
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000

SMOTE ROC-AUC Score: 0.9795


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

### Final Lab Synthesis: Resampling Evaluation Metrics Analysis

1. **Why Baseline Metrics Fail:** Accuracy is highly misleading for fraud detection. The baseline model achieved 96% accuracy but missed 40% of actual fraud (Recall = 0.60). In financial fraud detection, Recall for Class 1 (Fraud) is the priority metric.

2. **Comparison of Resampling Methods (Oversample, Undersample, SMOTE):**
   - **All three methods** drastically improved the model's performance, jumping Class 1 Recall from 60% up to 95%, with a near-perfect ROC-AUC score of ~0.9795. 
   - This identical performance implies that the credit card fraud features hold an incredibly strong, explicit signal.


3. **The Tradeoff (Precision vs. Recall):** In all balanced models, Precision dropped to 58%. This means the model creates more False Positives (flagging legitimate cards). In bank infrastructure, sending a text verification to a customer is a cheap and acceptable compromise compared to missing a major fraud attack.


4. **Production Recommendation:** Even though their scores are identical, **SMOTE** or **Oversampling** is preferred over Undersampling. Undersampling threw away roughly 800,000 rows of valid data to balance the dataset, which can cause severe data loss and underfitting on more complex, real-world data distributions.
